<a href="https://colab.research.google.com/github/bmf87/pydata_copilot/blob/main/notebooks/quantize_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 552 - Generative AI
* Capstone Project Title: PyData Copilot
* Objective: Direct Preference Optimization (DPO) of Qwen2.5-Coder-7B-Instruct
* Author: Brett Favro

In [ ]:
!pip install -q -U huggingface-hub llama-cpp-python[server] transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 39.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

## Clone llama.cpp
* Includes convert.py

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 85324, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 85324 (delta 45), reused 25 (delta 25), pack-reused 85239 (from 3)
Receiving objects: 100% (85324/85324), 338.88 MiB | 42.81 MiB/s, done.
Resolving deltas: 100% (61572/61572), done.
/content/llama.cpp


In [ ]:
# CMake build
!cmake -B build
!cmake --build build --config Release -j

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

## Convert & Quantize Model
* Download HF repo
* Convert from HF layout to half-precision (FP16)
* Quantize to q4_k_m (INT4)

### Hugging Face Format

HF $\rightarrow$ FP16 GGUF &nbsp;

HF Format: safetensors + sidecar files. Intended for training/fine-tuning, and framework inference
- model.safetensors - tensor weights in safetensors format. Raw tensors in a non-executable binary.
- config.json - architecture hyperparameters
- tokenizer.json / tokenizer.model / tokenizer_config.json

In [ ]:
fp16_model_dir = "qwen2.5-dpo-local"

local_dir = snapshot_download(
    #repo_id="bfavro73/qwen2.5-coder-1.5b-pandas-dpo-aligned",
    repo_id="bfavro73/qwen2.5-coder-7b-pandas-dpo-aligned",
    repo_type="model",               # default
    local_dir=fp16_model_dir,        # /content/llama.cpp/qwen2.5-dpo-local
)
print(local_dir)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

/content/llama.cpp/qwen2.5-dpo-local


In [ ]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py "{fp16_model_dir}" \
    --outtype f16 \
    --outfile qwen2.5-coder-7b-pandas-dpo-aligned-f16.gguf

/content/llama.cpp
INFO:hf-to-gguf:Loading model: qwen2.5-dpo-local
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,             torch.bfloat16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {18944, 3584}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {512}
INFO:hf-to-gguf:blk.0

## Generic GPT Unified Format

FP16 GGUF $\rightarrow$ Q4_K_M &nbsp;

GGUF Format: a binary self-contained format designed for quick loading / saving of models. Stores quantized weights optimized for lower-resource inference. GGUF supports nultiple levels of quantization:
- INT8: provides a balance between accuracy and model size.
- INT4: best when running the model on memory-constrained devices.



In [ ]:
%cd /content/llama.cpp

# Quantize to q4_k_m
!./build/bin/llama-quantize qwen2.5-coder-7b-pandas-dpo-aligned-f16.gguf \
    qwen2.5-coder-7b-pandas-dpo-aligned-q4_k_m.gguf Q4_K_M

/content/llama.cpp
main: build = 8495 (7cadbfce1)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'qwen2.5-coder-7b-pandas-dpo-aligned-f16.gguf' to 'qwen2.5-coder-7b-pandas-dpo-aligned-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from qwen2.5-coder-7b-pandas-dpo-aligned-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:    

In [ ]:
!ls -la /content/llama.cpp/*.gguf

-rw-r--r-- 1 root root 15237853056 Mar 24 03:14 /content/llama.cpp/qwen2.5-coder-7b-pandas-dpo-aligned-f16.gguf
-rw-r--r-- 1 root root  4683073408 Mar 24 03:19 /content/llama.cpp/qwen2.5-coder-7b-pandas-dpo-aligned-q4_k_m.gguf


In [ ]:
# Upload GGUF to HF repo

HF_TOKEN = userdata.get('HF_TOKEN') # get token from Colab secrets
api = HfApi(token=HF_TOKEN)
repo_id = "bfavro73/qwen2.5-coder-7b-pandas-dpo-aligned"
gguf_path = "/content/llama.cpp/qwen2.5-coder-7b-pandas-dpo-aligned-q4_k_m.gguf"
gguf_filename = gguf_path.split("/")[-1]

api.upload_file(
    path_or_fileobj=gguf_path,
    path_in_repo=gguf_filename,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Add q4_k_m GGUF (llama.cpp)"
)

print(f"GGUF model ready at: {repo_id}/{gguf_filename}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s-dpo-aligned-q4_k_m.gguf:   0%|          | 1.10MB / 4.68GB            

GGUF model ready at: bfavro73/qwen2.5-coder-7b-pandas-dpo-aligned/qwen2.5-coder-7b-pandas-dpo-aligned-q4_k_m.gguf
